In [ ]:
# Numbers
import numpy as np
import matplotlib.colors as mcolors
from matplotlib import colormaps

# Shapes
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon, mapping

# Data
from sqlalchemy import create_engine, text
import pandas as pd
from config import DATABASE_URL

# Maps
import h3
import folium

# --- ISOCHRONE MAP SETTINGS ---

origin_city_state = "Los Angeles, CA"   # Any in Continental United States
isolayer_increment = 500                # Miles
isochrone_resolution = 4                # 3-low, 4-med, 5-high, 6-very high

map_visual_settings = {
    "map_tile_theme": "OpenStreetMap",        # light default
    # "map_tile_theme": "CartoDB positron",     # light minimal
    # "map_tile_theme": "CartoDB dark_matter",    # dark minimal

    # "isochrone_color_scheme": "RdYlGn_r",       # dark tiles - red, yellow, green
    # "isochrone_color_scheme": "Spectral_r",   # dark tiles - rainbow
    "isochrone_color_scheme": "turbo",        # light tiles - rainbow
    # "isochrone_color_scheme": "viridis_r",    # light tiles - purp, blue, green, yellow

    "isochrone_opacity": 0.5,
    
    "isochrone_outline_weight": 3,
}

def print_settings(on:bool) -> None:
    if on:
        print(f"""
        🌎🌏 Generating Isochrone Map 🌎🌏

        Origin:         {origin_city_state}, USA
        Iso-Layers:     Increments of {isolayer_increment} Miles
        Iso-Resolution: {isochrone_resolution}
        Tiles:          {map_visual_settings["map_tile_theme"]}
        Iso-Colors:     {map_visual_settings["isochrone_color_scheme"]}
        """)

print_settings(on=True)

In [ ]:
# --- SQL & Data ---

ENGINE = create_engine(DATABASE_URL)
SQL_BATCH_SIZE = 5750

CITY_LAT_LNG = pd.read_csv("datasets/city_lat_long.csv") # Contains all US Cities, Towns, etc.

# --- Sub-Functions ---

def identify_origin_cell(origin: str, resolution: int, lat_lngs: pd.DataFrame = CITY_LAT_LNG) -> str:
    """
    origin must be a valid CONUS City: "{Capitolized CONUS City}, {All Caps 2-Character State}"
    Ex:
        >>> identify_origin_cell("Portland, OR", 5)
        '8528f003fffffff' 
        👆 H3 cell ID with resolution: 5 (hex string)
    """
   
    origin_state = origin[-2:]
    origin_city = origin[:-4]

    # - 1 - find origin coordinates from origin name using dataset of all CONUS cities
    origin_df = lat_lngs.loc[
        (lat_lngs["city"] == origin_city) &
        (lat_lngs["state"] == origin_state)
    ].reset_index(drop=True)
    
    origin_latitude = origin_df.loc[0, "latitude"]
    origin_longitude = origin_df.loc[0, "longitude"]

    # - 2 - find origin cell from coordinates
    origin_cell = h3.latlng_to_cell(
        lat=origin_latitude,
        lng=origin_longitude,
        res=resolution
    )    

    return origin_cell

def get_avg_dist_of_neighbors(cell: str, lookup_df: pd.DataFrame) -> float:
    neighbors = h3.grid_ring(cell, 1)

    avg = (
        lookup_df.loc[
            lookup_df["cell_id"].isin(neighbors),
            "driving_distance_m"
        ]
        .mean()
    )

    return avg

def query_nodes(origin_cell: str, resolution: int, isolayer_increment) -> pd.DataFrame:
    # - 1 - load node cache with road snapped points for resolution
    nodes_df = pd.read_csv(f"datasets/initial_res_{resolution}_points.csv")

    # Road snapped center point of origin cell
    origin_road_snap_latitude = float(
        nodes_df.loc[nodes_df["CellID"] == origin_cell, "RSLatitude"].iloc[0]
    )
    origin_road_snap_longitude = float(
        nodes_df.loc[nodes_df["CellID"] == origin_cell, "RSLongitude"].iloc[0]
    )

    # Prep dataframe for transit distance query
    nodes_df = nodes_df[["CellID", "RSLatitude", "RSLongitude"]].rename(
        columns={
            "CellID": "cell_id",
            "RSLatitude": "lat",
            "RSLongitude": "lng",
        }
    )
    
    # - 2 - batch query transit distances for all nodes
    with ENGINE.begin() as conn:
        # 1️⃣ Create temp table for raw destination points
        conn.execute(text("""
            CREATE TEMP TABLE destination_points (
                cell_id TEXT,
                lat DOUBLE PRECISION,
                lng DOUBLE PRECISION
            ) ON COMMIT DROP;
        """))

        nodes_df.to_sql(
            "destination_points",
            conn,
            if_exists="append",
            index=False,
            method="multi"
        )

        # 2️⃣ Materialize nearest node for each destination
        conn.execute(text("""
            CREATE TEMP TABLE destinations AS
            SELECT
                d.cell_id,
                r.target AS node_id
            FROM destination_points d
            JOIN LATERAL (
                SELECT target
                FROM routing_roads
                ORDER BY geom <-> ST_Transform(
                    ST_SetSRID(ST_Point(d.lng, d.lat), 4326),
                    3857
                )
                LIMIT 1
            ) r ON true;
        """))

        conn.execute(text("""
            CREATE INDEX ON destinations (node_id);
        """))

        # 3️⃣ Fetch distinct node_ids
        node_ids = pd.read_sql(
            "SELECT DISTINCT node_id FROM destinations",
            conn
        )["node_id"].tolist()

        print(f"Routing origin to {len(node_ids)} destinations")

        # 4️⃣ Prepare batched Dijkstra SQL
        batch_sql = text("""
        WITH origin AS (
            SELECT source AS id
            FROM routing_roads
            ORDER BY geom <-> ST_Transform(
                ST_SetSRID(ST_Point(:lng1, :lat1), 4326),
                3857
            )
            LIMIT 1
        )
        SELECT
            r.end_vid AS node_id,
            SUM(r.cost) AS driving_distance_m
        FROM pgr_dijkstra(
            'SELECT id, source, target, cost,
                    COALESCE(reverse_cost, cost) AS reverse_cost
            FROM routing_roads',
            (SELECT id FROM origin),
            :node_array,
            true
        ) r
        GROUP BY r.end_vid;
        """)

        # 5️⃣ Run in batches
        all_results = []

        for i in range(0, len(node_ids), SQL_BATCH_SIZE):
            batch = node_ids[i:i+SQL_BATCH_SIZE]

            print(f"Batch {i//SQL_BATCH_SIZE + 1} ({len(batch)} nodes)")

            result = pd.read_sql(
                batch_sql,
                conn,
                params={
                    "lat1": origin_road_snap_latitude,
                    "lng1": origin_road_snap_longitude,
                    "node_array": batch
                }
            )

            all_results.append(result)

        # 6️⃣ Combine batch results
        result_df = pd.concat(all_results, ignore_index=True)

        # 7️⃣ Load mapping: cell_id -> node_id
        cell_to_node = pd.read_sql(
            "SELECT cell_id, node_id FROM destinations",
            conn
        )

        # 8️⃣ Attach distances to each cell via node_id
        cell_distances = cell_to_node.merge(
            result_df,
            on="node_id",
            how="left"
        )

        # 9️⃣ Merge back onto nodes dataframe
        nodes_df = nodes_df.merge(
            cell_distances[["cell_id", "driving_distance_m"]],
            on="cell_id",
            how="left"
        )

        print("Done.")

    while nodes_df["driving_distance_m"].isnull().any():
        mask = nodes_df["driving_distance_m"].isnull()

        nodes_df.loc[mask, "driving_distance_m"] = (
            nodes_df.loc[mask, "cell_id"]
            .apply(lambda cell: get_avg_dist_of_neighbors(cell, nodes_df))
        )

    nodes_df["driving_distance_miles"] = nodes_df["driving_distance_m"]/1608.344

    nodes_df["driving_distance_days"] = (
        pd.to_numeric(nodes_df["driving_distance_miles"], errors="coerce")
        / isolayer_increment
    ).apply(np.ceil)

    return nodes_df

def get_unique_day_polygon(df: pd.DataFrame, transit_day: int) -> MultiPolygon:
    filtered_df = df.loc[df["driving_distance_days"] == transit_day]
    h3_indexes = filtered_df["cell_id"].unique()

    polygons = []

    for h in h3_indexes:
        boundary = h3.cell_to_boundary(h)
        polygons.append(Polygon(boundary))

    merged = unary_union(polygons)

    if isinstance(merged, Polygon):
        merged = MultiPolygon([merged])

    return merged

def flip_coords(geometry: Polygon | MultiPolygon) -> Polygon | MultiPolygon:
    """Swap (x, y) → (y, x) recursively for Polygon/MultiPolygon"""
    if geometry.is_empty:
        return geometry

    geometry = geometry.buffer(0.001).buffer(-0.001) # remove multipolygon internal artifacts

    if geometry.geom_type == "Polygon":
        exterior = [(y, x) for x, y in geometry.exterior.coords]
        interiors = [[(y, x) for x, y in i.coords] for i in geometry.interiors]
        return Polygon(exterior, interiors)
    
    elif geometry.geom_type == "MultiPolygon":
        return MultiPolygon([flip_coords(p) for p in geometry.geoms])
    
    else:
        return geometry

def set_isochrone_geometries(map_nodes_df: pd.DataFrame) -> dict:
    # - 1 - define each isochrone color based on min/max of transit distances
    map_nodes_df = map_nodes_df.dropna(subset=["driving_distance_days"])

    unique_transit_days = map_nodes_df["driving_distance_days"].unique()

    # - 2 - define multipolygons for each isolayer
    geometries = {
        day: get_unique_day_polygon(map_nodes_df, day)
        for day in unique_transit_days
    }

    for day in geometries:
        geometries[day] = flip_coords(geometries[day])  # swap coords for Folium

    return geometries 

def innitalize_map(isochrone_geometries: dict, map_visual_settings:dict) -> folium.Map:

    cmap = colormaps[map_visual_settings["isochrone_color_scheme"]]

    norm_transit_days = mcolors.Normalize(vmin=1, vmax=max(isochrone_geometries))
    
    # - 1 - define blank map
    m = folium.Map(
        location=(40, -100), 
        zoom_start=5, 
        tiles=map_visual_settings["map_tile_theme"]
        )
    # - 2 - add isochrones to map
    for day, polygons in isochrone_geometries.items():

        isochrone_color = mcolors.to_hex(cmap(norm_transit_days(day)))
        folium.GeoJson(
            data={
                "type": "Feature",
                "geometry": mapping(polygons),
                "properties": {}
            },
            style_function=lambda feature, col=isochrone_color: {
                "fill": True,
                "color": col,
                "fill_opacity": map_visual_settings["isochrone_opacity"],
                "weight": map_visual_settings["isochrone_outline_weight"]
            }
        ).add_to(m)    
    return m

# --- Main Function ---
def generate_isochrone_map():
    origin_cell = identify_origin_cell(origin_city_state, isochrone_resolution)

    map_nodes_df = query_nodes(origin_cell, isochrone_resolution, isolayer_increment)

    isochrone_geometries = set_isochrone_geometries(map_nodes_df)

    map = innitalize_map(isochrone_geometries, map_visual_settings)

    return map

In [ ]:
# --- Main Function Call ---
m = generate_isochrone_map()

m